In [39]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
table {margin-left:0 !important; margin-right: auto;}
</style>
"""))

# 생성형 AI 평가
- 첫번째 체인 : 국가명 → 해당 국가에서 가장 유명한 음식
- 두번째 체인 : 음식 → 해당 음식의 레시피
- 최종 체인 : 국가명 → 해당 국가에서 가장 유명한 음식의 레시피

### 파일명 : 생성형AI.ipynb, 생성형AI.html, 실행화면.png를 압축하여 "본인이름_생성형AI.zip으로 메일 첨부해주세요.

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser

In [7]:
llm = ChatOllama(
    model='exaone3.5:2.4b',
    temperature=0.2,
    top_k=40,
    top_p=0.9,
)

outputParser = StrOutputParser()

In [14]:
first_prompt = PromptTemplate(
    template='{나라}에서 가장 유명한 음식은 무엇인가요? 음식의 이름만 알려주세요.',
    input_variables = ['나라']
)

In [15]:
first_chain = first_prompt | llm | outputParser

In [16]:
second_prompt = PromptTemplate(
    template='{음식}의 레시피는 무엇인가요?',
    input_variables = ['음식']
)

In [17]:
second_chain = second_prompt | llm | outputParser

In [18]:
final_chain = first_chain | second_chain

In [21]:
result = final_chain.invoke({'나라':'미국'})

In [22]:
print(result)

피자 레시피는 다양한 스타일과 재료에 따라 크게 달라질 수 있지만, 가장 기본적인 토마토 소스 기반 피자 레시피를 소개해 드리겠습니다:

### 기본 토마토 소스 피자 레시피

#### 재료:
- **밀가루**: 250g (또는 원하는 양)
- **물**: 100ml (또는 필요한 만큼)
- **드라이 이스트**: 7g (또는 활성 건조 효모)
- **소금**: 약간 (약 1 tsp)
- **설탕**: 약간 (약 1 tsp)
- **올리브 오일**: 2 tbsp
- **토마토 소스**: 300g (또는 원하는 양)
- **치즈**: 모짜렐라 치즈 약 150g (또는 다른 종류의 치즈)
- **피망**: 취향에 따라 (선택 사항)
- **페퍼oncino**: 약간 (선택 사항)
- **올리브 슬라이스**: 장식용 (선택 사항)

#### 조리 과정:
1. **반죽 만들기**:
   - 큰 볼에 밀가루, 소금, 설탕, 드라이 이스트를 넣고 잘 섞습니다.
   - 따뜻한 물(약 110°F/43°C)을 천천히 첨가하며 반죽을 시작합니다. 손으로 치대어 부드럽고 탄력이 있는 반죽이 될 때까지 약 10분간 치댑니다.
   - 반죽이 완성되면 깨끗한 그릇에 옮겨 뚜껑을 덮고 약 1시간 동안 발효시킵니다 (온도 약 75-80°F/24-27°C).

2. **발효 후 성형**:
   - 발효된 반죽을 부드럽게 펴서 원하는 크기의 피자 도우를 만듭니다.
   - 도우를 원형으로 펴고, 중앙에 토마토 소스를 고르게 펴발라줍니다.

3. **치즈와 토핑 추가**:
   - 펴놓은 도우 위에 모짜렐라 치즈를 골고루 뿌립니다.
   - 원하는 토핑을 추가합니다 (피망, 페퍼oncino 등).

4. **굽기**:
   - 오븐을 약 450°F (230°C)로 예열합니다.
   - 피자를 오븐 트레이에 올려놓고 약 12-15분간 굽습니다 (오븐 종류에 따라 조절).
   - 치즈가 녹고 토핑이 익을 때까지 굽습니다.

5. **완성 및 즐기기**:
   - 완성된 피자를 꺼내어 식힌 후

---

In [23]:
from pydantic import BaseModel, Field

In [29]:
first_prompt = PromptTemplate(
    template='{나라}에서 가장 유명한 음식은 무엇인가요? 음식의 이름만 알려주세요.',
    input_variables = ['나라']
)

class foodDetail(BaseModel):
    food:str = Field(description='해당 국가의 가장 유명한 음식')
        
structedllm = llm.with_structured_output(foodDetail)

food_chain = first_prompt|structedllm

food_chain.invoke({'나라': '이탈리아'})

foodDetail(food='피자')

In [36]:
second_prompt = PromptTemplate(
    template='{음식}의 레시피는 무엇인가요? 섭씨와 ml, g의 단위를 사용해주세요',
    input_variables = ['음식']
)

class recipeDetail(BaseModel):
    recipe:str = Field(description='해당 음식의 가장 유명한 레시피')
        
structedllm = llm.with_structured_output(recipeDetail)

recipe_chain = second_prompt|structedllm

recipe_chain.invoke({'음식': '김치'})

recipeDetail(recipe='김치 레시피\n\n**재료**:\n- 배추 2포기\n- 고춧가루 100g\n- 마늘 3쪽\n- 생강 1작은술\n- 소금 약간\n- 멸치액젓 50ml\n- 설탕 약간\n- 참기름 약간\n- 파 다진 것 약간\n- 후추 약간\n\n**만드는 방법**:\n1. **배추 준비**:\n   - 배추를 깨끗이 씻어 물기를 제거하고, 뿌리 부분을 잘라내고 잎을 펴서 준비합니다.\n2. **양념 만들기**:\n   - 볼에 고춧가루, 마늘, 생강, 멸치액젓, 설탕, 참기름을 넣고 잘 섞습니다.\n3. **배추에 양념 바르기**:\n   - 준비한 배추 잎에 양념을 골고루 바릅니다.\n4. **밀폐 용기에 담기**:\n   - 배추를 비닐 랩이나 밀폐 용기에 넣고, 손으로 눌러 공기를 제거한 후 밀봉합니다.\n5. **숙성**:\n   - 실온에서 약 12-24시간 숙성시킵니다. 냉장 보관 시 2-3일 정도 더 숙성하면 맛이 깊어집니다.\n\n**단위 변환 예시**:\n- **ml**: 멸치액젓은 약 50ml 정도가 적당합니다.\n- **g**: 고춧가루는 약 100g을 사용합니다.\n\n**주의사항**:\n- 배추의 크기와 양에 따라 양념의 양을 조절할 수 있습니다.')

In [38]:
final_chain = food_chain | recipe_chain

country = input('국가명 입력 : ')

result = final_chain.invoke({'나라': country})

print(result.recipe)

국가명 입력 : 대만
마라탕 레시피

**재료**:
- 돼지고기 (약 500g): 얇게 썰어 준비
- 소고기 (약 300g): 얇게 썰어 준비
- 두부 (약 200g): 깍둑썰기
- 배추잎 (약 1kg): 깨끗이 씻어 물기 제거
- 마늘 (3쪽): 다진 것
- 생강 (1쪽): 다진 것
- 고춧가루 (5큰술): 취향에 따라 조절
- 간장 (3큰술): 맛 조절
- 설탕 (1큰술): 단맛 조절
- 참기름 (1큰술)
- 소금 (약간)
- 후추 (약간)

**조리법**:
1. **양념 준비**: 볼에 간장, 설탕, 고춧가루를 섞어 기본 양념을 만듭니다.
2. **고기 조리**: 냄비에 기름을 두르고 돼지고기와 소고기를 볶아 익힙니다. 이때 마늘과 생강을 함께 넣어 향을 내줍니다.
3. **배추 조리**: 배추잎을 물에 담가 살짝 데쳐서 물기를 제거합니다. 이 과정에서 배추의 수분을 적절히 유지하는 것이 중요합니다.
4. **조합**: 볶아낸 고기와 데친 배추를 큰 그릇에 담고, 양념을 골고루 뿌려 섞습니다.
5. **마무리**: 두부를 추가하고 모든 재료가 잘 어우러지도록 잠시 더 섞어줍니다. 마지막으로 참기름과 소금, 후추로 간을 맞춥니다.

**단위**:
- 재료 무게: g (예: 돼지고기 500g)
- 부피: ml (예: 양념 재료를 섞을 때 사용하는 물의 양)
- 조리 시간: 시간 단위 (예: 10분)

**주의사항**:
- 재료의 신선도가 맛에 큰 영향을 미치므로 신선한 재료를 사용하세요.
- 개인의 취향에 따라 양념 비율을 조절하세요.
